In [1]:
import sys
import numpy as np

# sys.path.append('../../../src/')
from Rain.Rain import Rain
# sys.path.pop()

from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-10 16:03:06.682510: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-10 16:03:07.982236: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
config = {
  "mode": {
      "type": "lazy",
      "params": {
          "num_of_workers": 1,
          "ips": ['127.0.0.1'],
          "ports": [50151],
          "subscription_id": "a7ef3688-af58-4835-953c-e51f219fbd0f", # Mostafa's ID
          # "subscription_id":'6e14c264-a7fc-4db4-a23a-d972c21a2d99', # Menna's ID
          # "subscription_id": '82305756-d4a0-442d-8e73-625e1ced2113', # Nada's ID
          "location": 'eastus',
          'vm_size': 'Standard_B2ms'
      }
    },
  "temp_data_path": "../../../",
  "partitions": 1,
  "iterations": 3,
  "chunk_size": 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 5,
    "batch_size": 16,
  }
}

In [3]:
def get_train_data():
    return np.load("../../../data/breast_cancer/train_data.npy"), np.load(
        "../../../data/breast_cancer/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/breast_cancer/test_data.npy"), np.load(
        "../../../data/breast_cancer/test_labels.npy"
    )



In [4]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 30
    num_labels = 2
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [5]:
X_train, y_train = get_train_data()
X_train = np.reshape(X_train, [-1, 30])
y_train = to_categorical(y_train)

In [6]:
model = create_model()
rain = Rain(config, model)

2023-07-10 16:03:10,894 [INFO] [Rain] Rain is initialized
2023-07-10 16:03:10,897 [INFO] [Provisioner] Creating coordinator
2023-07-10 16:03:10,900 [INFO] [Coordinator] Coordinator is initialized
2023-07-10 16:03:10,902 [INFO] [LazyProvisioner] Provisioner is initialized


In [7]:
# model = rain.train(X_train, y_train, strategy='async')

In [8]:
# X_test, y_test = get_test_data()
# X_test = np.reshape(X_test, [-1, 30])
# y_test = to_categorical(y_test)
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

In [9]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-10 16:03:11,422 [INFO] [Provisioner] provisioner is serving
2023-07-10 16:03:11,423 [INFO] [Provisioner] Starting coordinator
2023-07-10 16:03:11,433 [INFO] [Coordinator] coordinator is serving
2023-07-10 16:03:11,458 [INFO] [LazyProvisioner] Creating 1 workers
2023-07-10 16:03:11,461 [INFO] [Provisioner] [Created workers]: IPs : ['127.0.0.1'], ports: [50151], statuses: [1], IDs : [1]
2023-07-10 16:03:11,465 [INFO] [DividerProxy] Training Started
2023-07-10 16:03:11,473 [INFO] [DeepLearning] Starting iteration 1/3
2023-07-10 16:03:11,542 [INFO] [DividerAmbassador] Executing one iteration of federated learning on 127.0.0.1:50151
2023-07-10 16:03:11,608 [INFO] [DividerAmbassador] divider begins executing iteration1 for worker1
2023-07-10 16:03:17,579 [INFO] [DeepLearning] Iteration 1/3 complete.
2023-07-10 16:03:17,581 [INFO] [DeepLearning] Starting iteration 2/3
2023-07-10 16:03:17,625 [INFO] [DividerAmbassador] Executing one iteration of federated learning on 127.0.0.1:50151
20

In [10]:
X_test, y_test = get_test_data()
X_test = np.reshape(X_test, [-1, 30])
y_test = to_categorical(y_test)
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

8/8 [==============================] - 1s 3ms/step - loss: 0.2144 - accuracy: 0.9211

Test accuracy: 92.1%


In [11]:
del rain

2023-07-10 16:03:22,189 [INFO] [Provisioner] Workers deleted
2023-07-10 16:03:22,193 [INFO] [Provisioner] provisioner stopped serving
